# Jalon 03 — M25 : contrat OpenAPI et tracabilite des appels

Ce notebook accompagne `tests/test_request_id.py`. Il sert a **explorer** avant de
figer les preuves dans des tests.

Deux questions :

1. **Le contrat est-il publie ?** Une regle metier qui n'apparait pas dans
   `/openapi.json` n'est pas opposable a un client tiers.
2. **Un appel est-il tracable ?** Sans identifiant de correlation, un incident en
   production ne se diagnostique pas.

## Prerequis

Depuis la racine du depot, sur la branche `jalon/02` :

```powershell
uv sync --frozen --extra dev
uv run jupyter lab
```

Aucun modele entrainee n'est necessaire : on n'appelle que `/health` et le
document OpenAPI, qui ne dependent pas du bundle.

In [15]:
import sys

from fastapi.testclient import TestClient

from indusense.api.main import app

# TestClient : un client HTTP en memoire. Aucun port reseau n'est ouvert,
# aucun serveur uvicorn a lancer. C'est ce qui rend ces tests rapides.
client = TestClient(app)

reponse = client.get("/health")
print(reponse.status_code, reponse.json())

200 {'status': 'ok'}


## 1. Le contrat publie

FastAPI **genere** `/openapi.json` a partir des decorateurs de route et des
schemas Pydantic. Ce document n'est jamais ecrit a la main : il reflete
mecaniquement le code. C'est ce qui le rend testable.

In [16]:
spec = client.get("/openapi.json").json()

print("Version OpenAPI :", spec["openapi"])
print("Routes exposees :", sorted(spec["paths"]))
print("Schemas publies :", sorted(spec["components"]["schemas"]))

Version OpenAPI : 3.1.0
Routes exposees : ['/health', '/predict-tabular', '/ready']
Schemas publies : ['HTTPValidationError', 'PredictionResponse', 'SensorReading', 'TabularPredictionRequest', 'ValidationError']


### Attention — ne tester que les routes qui existent

Au jalon 03, `main.py` expose **trois** routes : `/health`, `/ready` et
`/predict-tabular`.

Une route comme `/predict-image` arrive plus tard dans le parcours. L'inclure
ici produirait un echec permanent, ce qui n'est pas une specification : c'est un
test casse. La cellule suivante le verifie explicitement.

In [17]:
ROUTES_ATTENDUES = ("/health", "/ready", "/predict-tabular")

for route in ROUTES_ATTENDUES:
    assert route in spec["paths"], f"Route absente du contrat : {route}"

print("OK — les", len(ROUTES_ATTENDUES), "routes du jalon 03 sont publiees.")

# Verification du piege : cette route n'existe pas encore.
print("/predict-image present ?", "/predict-image" in spec["paths"])

OK — les 3 routes du jalon 03 sont publiees.
/predict-image present ? False


### La contrainte metier des 7 releves

`schemas.py` declare :

```python
readings: list[SensorReading] = Field(..., min_length=7)
```

Pourquoi 7 ? `add_temporal_features` calcule une moyenne glissante sur 6 points
apres un `shift(1)`. Il faut donc 6 lignes d'historique **plus** la ligne
courante pour produire une seule prediction exploitable.

Cette regle doit etre **visible** dans la documentation : un client qui genere
son code depuis le contrat doit connaitre la contrainte sans lire notre source.

In [18]:
schema = spec["components"]["schemas"]["TabularPredictionRequest"]
readings = schema["properties"]["readings"]

readings

{'items': {'$ref': '#/components/schemas/SensorReading'},
 'type': 'array',
 'minItems': 7,
 'title': 'Readings'}

In [19]:
# Pydantic v2 traduit `min_length` (liste) en `minItems` (JSON Schema).
# C'est ce que Swagger affiche, et ce qu'un generateur de client lira.
assert readings["minItems"] == 7

print("OK — la contrainte des 7 releves est publiee dans le contrat.")

OK — la contrainte des 7 releves est publiee dans le contrat.


## 2. La tracabilite : X-Request-ID

`main.py` installe un middleware (lignes 66 a 73) :

```python
@app.middleware("http")
async def add_request_id(request: Request, call_next):
    request_id = request.headers.get("X-Request-ID", str(uuid.uuid4()))
    response = await call_next(request)
    response.headers["X-Request-ID"] = request_id
    return response
```

Deux comportements en une ligne : si le client fournit un identifiant, il est
**repris tel quel** ; sinon, le serveur en **genere** un.

Le premier cas est le plus important en production : il permet de relier la
trace du client a celle du serveur pour un meme appel.

In [20]:
reponse = client.get("/health", headers={"X-Request-ID": "abc-123"})

print("Renvoye :", reponse.headers["X-Request-ID"])
assert reponse.headers["X-Request-ID"] == "abc-123"

Renvoye : abc-123


In [21]:
import uuid

reponse = client.get("/health")
identifiant = reponse.headers["X-Request-ID"]

print("Genere :", identifiant)

# uuid.UUID(...) leve ValueError si le format est invalide.
# Pas besoin d'assert : l'exception suffit a faire echouer le test.
uuid.UUID(identifiant)
print("Format UUID valide.")

Genere : 2fa25653-f3fa-4c25-b62c-eab594e7fc22
Format UUID valide.


### Deux pieges a verifier

**Unicite.** Un identifiant calcule une seule fois au demarrage passerait les
deux tests precedents tout en etant inutile. Il faut verifier que deux appels
produisent deux valeurs differentes.

**Reponses d'erreur.** Un middleware doit s'appliquer a *toutes* les reponses.
Si les erreurs n'etaient pas tracees, les seuls appels non diagnosticables
seraient precisement ceux qui posent probleme.

In [22]:
premier = client.get("/health").headers["X-Request-ID"]
second = client.get("/health").headers["X-Request-ID"]

print("Appel 1 :", premier)
print("Appel 2 :", second)
assert premier != second
print("OK — un identifiant distinct par appel.")

Appel 1 : b78f1959-47b2-42ce-b243-42ce3760f1da
Appel 2 : d6e55baa-c47e-4943-834f-3f696040da92
OK — un identifiant distinct par appel.


In [23]:
# Aucune cle API : require_api_key leve une HTTPException 401.
reponse = client.post("/predict-tabular", json={})

print("Statut :", reponse.status_code)
print("X-Request-ID :", reponse.headers["X-Request-ID"])

uuid.UUID(reponse.headers["X-Request-ID"])
print("OK — meme une erreur est tracable.")

Statut : 401
X-Request-ID : 0ac471f1-2a38-4a80-b86d-5e12cd5c66c6
OK — meme une erreur est tracable.


## 3. Passer du notebook aux tests

L'exploration ci-dessus est jetable. Les preuves, elles, doivent tourner en CI a
chaque commit. Elles sont figees dans `tests/test_request_id.py`.

Depuis la racine du depot :

```powershell
uv run pytest tests/test_request_id.py -q
```

Sept tests : trois sur le contrat OpenAPI, quatre sur la propagation de
l'identifiant.

In [24]:
import subprocess

resultat = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_request_id.py", "-q"],
    capture_output=True,
    text=True,
)
print(resultat.stdout[-1500:])


no tests ran in 0.01s



## A retenir

| Constat | Consequence |
|---|---|
| `/openapi.json` est genere depuis le code | Le contrat ne peut pas deriver de l'implementation |
| `min_length=7` devient `minItems: 7` | La regle metier est opposable a un client tiers |
| Le middleware couvre toutes les reponses | Les erreurs restent diagnosticables |
| Un identifiant par appel | La correlation dans les logs est possible |

**Question ouverte pour le jalon suivant** : l'identifiant est pose dans
l'en-tete de reponse, mais il n'apparait pas encore dans les logs applicatifs.
Sans cela, la correlation reste theorique. C'est l'objet du M26.